# Задача 3. Ймовірності та розподіли

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from math import comb

## Частина 1. Формула повної ймовірності та Байєса

In [2]:
pA, pB = 0.9, 0.1
pE_A, pE_B = 0.8, 0.3
a = pE_A * pA                 # P(E ∩ A)
b = pE_A * pA + pE_B * pB     # P(E)
c = pE_B * pB / b             # P(B | E)
print(f"(a) P(E and A) = {a:.3f}\n(b) P(E) = {b:.3f}\n(c) P(B|E) = {c:.4f}")

(a) P(E and A) = 0.720
(b) P(E) = 0.750
(c) P(B|E) = 0.0400


(a) P(E∩A)=P(E|A)P(A)=0.8·0.9=0.72. (b) За формулою повної ймовірності P(E)=0.72+0.3·0.1=0.75. (c) За Байєсом P(B|E)=0.03/0.75=0.04 — серед тих, хто склав, лише 4% були погано підготовлені.

## Частина 2. Біноміальний розподіл (n=5, p=0.6)

In [3]:
X = stats.binom(5, 0.6)
print("(a) P(0) =", round(X.pmf(0), 4))
print("(b) P(2) =", round(X.pmf(2), 4))
print("(c) P(>=3) =", round(X.sf(2), 4))

(a) P(0) = 0.0102
(b) P(2) = 0.2304
(c) P(>=3) = 0.6826


## Частина 3. Акції

In [4]:
mu, var = 0.03, 0.02
sd = np.sqrt(var)
N = stats.norm(mu, sd)
print("i)  P(X>0.01)  =", round(N.sf(0.01), 4))
print("ii) P(X>=0.01) =", round(N.sf(0.01), 4), "(для неперервної величини те саме)")
print("iii) P(X<0.01) =", round(N.cdf(0.01), 4))
print("iv) P(0.01<X<0.03) =", round(N.cdf(0.03) - N.cdf(0.01), 4))

# (b) портфель: 0.4 капіталу порівну у 10 акцій, 0.6 у безризиковий актив
rf = 0.01
w = 0.4 / 10
mu_p = 10 * w * mu + 0.6 * rf
var_p = 10 * w**2 * var
print("Портфель ~ Normal(", round(mu_p, 4), ",", round(var_p, 6), "), sd =", round(np.sqrt(var_p), 4))

i)  P(X>0.01)  = 0.5562
ii) P(X>=0.01) = 0.5562 (для неперервної величини те саме)
iii) P(X<0.01) = 0.4438
iv) P(0.01<X<0.03) = 0.0562
Портфель ~ Normal( 0.018 , 0.00032 ), sd = 0.0179


(b) Лінійна комбінація незалежних нормальних величин нормальна: $R_p\sim N(\mu_p,\sigma_p^2)$, де $\mu_p=0.4\cdot0.03+0.6\cdot0.01=0.018$, $\sigma_p^2=10\cdot(0.04)^2\cdot0.02=0.00032$ (σ≈0.0179). Диверсифікація різко знизила ризик (одна акція: σ≈0.141).

(c) Для двох активів $Var(aX+bY)=a^2\sigma_X^2+b^2\sigma_Y^2+2ab\,\rho\,\sigma_X\sigma_Y$. Додатна кореляція збільшує дисперсію портфеля (диверсифікація працює гірше), від'ємна — зменшує; при незалежності доданок з ρ зникає.

## Частина 4. Tesla (TSLA)

In [5]:
p = yf.download("TSLA", period="2y", interval="1wk", auto_adjust=True, progress=False)["Close"].squeeze()
ret = p.pct_change().dropna()
print(len(ret), "тижневих доходностей, mean =", round(ret.mean(), 4), "std =", round(ret.std(), 4))
for t in [-0.02, -0.04]:
    emp = (ret < t).mean()
    nor = stats.norm.cdf(t, ret.mean(), ret.std())
    print(f"P(R < {t}): емпірична = {emp:.4f}, нормальна = {nor:.4f}")

104 тижневих доходностей, mean = 0.0079 std = 0.0726
P(R < -0.02): емпірична = 0.3365, нормальна = 0.3503
P(R < -0.04): емпірична = 0.2308, нормальна = 0.2546


Емпірична ймовірність падіння нижче −2% ≈ 33.7%, нижче −4% ≈ 23.1%; за нормальною моделлю — 35.0% і 25.5%. Результати близькі. Емпіричний метод не робить припущень про розподіл, але залежить від малої вибірки (104 точки) і не може оцінити рідкісні події поза діапазоном даних. Нормальна модель гладка й дає оцінку для будь-якого порога, але реальні доходності мають «важкі хвости» та асиметрію, тому ризик великих падінь вона може недооцінювати.

## Частина 5. Біноміальна модель

In [6]:
p_up = 0.55
B = lambda n, k: stats.binom.pmf(k, n, p_up)
print("i)  5 з 5   =", round(B(5, 5), 4))
print("ii) 3 з 3   =", round(B(3, 3), 4))
print("iii) 3 з 5  =", round(B(5, 3), 4))
print("iv) 10 з 15 =", round(B(15, 10), 4))
r_up, r_dn = 0.02, -0.01
Er = p_up * r_up + (1 - p_up) * r_dn
print("E(r) за період =", Er, " за 2 періоди =", 2 * Er)
print("E[S(2)] =", 10 * (1 + Er) ** 2)

i)  5 з 5   = 0.0503
ii) 3 з 3   = 0.1664
iii) 3 з 5  = 0.3369
iv) 10 з 15 = 0.1404
E(r) за період = 0.0065000000000000014  за 2 періоди = 0.013000000000000003
E[S(2)] = 10.130422499999998


(b) Періоди незалежні, тому очікувана дохідність за період E(r)=0.55·0.02+0.45·(−0.01)=0.0065; за два періоди ≈ 0.013 (проста сума), точніше (1.0065)²−1≈0.0130, тобто E[S(2)]≈10.13.

## Частина 6. Ймовірності для різних розподілів

In [7]:
dists = {"N(0,1)": stats.norm(), "t2": stats.t(2), "Exp(1)": stats.expon(), "chi2(3)": stats.chi2(3)}
tab = pd.DataFrame({
    name: {"a) P(1<X<1.5)": d.cdf(1.5) - d.cdf(1),
           "b) P(1<=X<1.5)": d.cdf(1.5) - d.cdf(1),
           "c) P(1<X)": d.sf(1),
           "d) P(X<=1.5)": d.cdf(1.5),
           "e) P(X=1.5)": 0.0}
    for name, d in dists.items()})
tab.round(4)

,"N(0,1)",t2,Exp(1),chi2(3)
a) P(1<X<1.5),0.0918,0.0751,0.1447,0.1190
b) P(1<=X<1.5),0.0918,0.0751,0.1447,0.1190
c) P(1<X),0.1587,0.2113,0.3679,0.8013
d) P(X<=1.5),0.9332,0.8638,0.7769,0.3177
e) P(X=1.5),0.0000,0.0000,0.0000,0.0000


Усі розподіли неперервні, тому P(X=1.5)=0 і строгість/нестрогість нерівностей (a та b) не впливає на результат.